**Riverside's traffic problem — a Monday morning emergency**

It's 8 a.m. Monday. Your editing assistant gateway (`06-llm-gateway.ipynb`) handled 600 requests over the weekend without a hiccup. The model quality is good, latency is acceptable, and your small team loves the tool.

Then a Slack message arrives from Marketing: _"We're onboarding all 5,000 Riverside authors next Friday. Traffic forecast: 1,000 requests per minute."_

You run the numbers: you're currently at **10 requests per minute**. That's a **100× shortfall** — and you have five days.

Your instinct is to ask for more GPUs. The infrastructure team says 6–8 weeks. The VP wants a launch date.

Here's what nobody tells you after the model training section: the bottleneck probably isn't hardware. It's the fact that your server recomputes attention from scratch on every decode step — idles the GPU between requests of different lengths — and pre-allocates KV cache memory for sequences 10× longer than the average request. These are **software bugs disguised as hardware problems**.

This notebook builds five software-level fixes — KV cache, continuous batching, PagedAttention, speculative decoding, and prefill/decode disaggregation — each compounding on the last, until 1,000 req/min is within reach without a single new GPU.


# LLM Inference Systems: From 10 to 1000 Requests Per Minute

| Part | Optimization              | Why this, right now?                                                                                                         | What it achieves                                                           |
| ---- | ------------------------- | ---------------------------------------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------- |
| 1    | KV cache                  | Your server recomputes attention over all prior tokens at _every_ decode step — that's O(S²) wasted work per request         | Eliminate the waste → 3–10× faster per-token latency, zero quality loss    |
| 2    | Continuous batching       | Faster tokens didn't help: the GPU still idles when a short request finishes and a longer one keeps running                  | Fill idle GPU slots → 2–3× higher throughput on the same hardware          |
| 3    | PagedAttention            | 100 concurrent KV caches each pre-allocated at max sequence length = 90% of GPU RAM sitting empty                            | Allocate by the page → 3× more concurrent users on the same GPU            |
| 4    | Speculative decoding      | Even with the above three, each new token forces one serial 7B-model forward pass — the model's native parallelism is wasted | Draft-then-verify → 1.5–3× faster generation with a small 70M helper model |
| 5    | Prefill vs. decode phases | Prompt processing and token generation hit the GPU in fundamentally different ways — treating them identically wastes both   | Understand the split → the basis for disaggregated production serving      |
| 6    | Toy → real bridge         | Our toy numbers must connect to the metrics a real ops team monitors in vLLM, TGI, or TensorRT-LLM                           | TTFT + TPOT → actionable production configuration                          |

---

## Prerequisite Bridge — From `04-llm/06-llm-gateway.ipynb` and `02-transformers`

| Foundation                                | Role in this notebook                                                           |
| ----------------------------------------- | ------------------------------------------------------------------------------- |
| LLM gateway (`06-llm-gateway.ipynb`)      | The serving infrastructure we're optimising — this chapter makes it faster      |
| KV cache (mentioned in `02-transformers`) | First described there as "an optimization"; this chapter builds it from scratch |
| Autoregressive generation                 | Every token is generated by a full forward pass through all layers              |

> **Prerequisites:** `learning/genai/04-llm/06-llm-gateway.ipynb` (gateway architecture) and `learning/genai/02-transformers/transformers.ipynb` (attention mechanism).


In [ ]:
import subprocess, sys

# Install torch/numpy/matplotlib/transformers only if missing
for pkg in ["torch", "numpy", "matplotlib", "transformers"]:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time
import random

# Use a GPU if available; all timing comparisons below are still valid on CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HAS_GPU = torch.cuda.is_available()
torch.manual_seed(42)

print(f"Device: {DEVICE}")
print(f"Riverside scenario: gateway serving 10 req/min → needs to handle 1,000 req/min")
print()

# Toy model config (GPT-2 small scale)
VOCAB_SIZE = 1000
D_MODEL = 256
N_HEADS = 8
N_LAYERS = 4
D_HEAD = D_MODEL // N_HEADS

---

## Part 1 — KV Cache: Never Recompute What You've Already Computed

In standard autoregressive generation, each new token requires a full attention computation over **all** previous tokens. For a 128-token prompt + 20-token response:

- Token 1: attention over 128 tokens (the prompt)
- Token 2: attention over 129 tokens
- Token 20: attention over 147 tokens

Each step recomputes keys and values for all previous positions — redundant work.

**KV cache:** compute K and V for each token once, store them, and reuse:

- Prefill: process the full prompt, store all K/V pairs in cache
- Decode: for each new token, compute only 1 new K/V pair; look up the rest from cache

#### #### Predict first

Without KV cache: each decode step runs attention over N positions.
With KV cache: each new token attends over how many _new_ positions?

1. **(a) 128 positions** — still needs to see all prompt tokens
2. **(b) 1 position** — only the new token needs new K/V
3. **(c) 1 new K/V pair computed, rest from cache** — 1 new computation + cache lookup


> **Intuition — it's just a cache:** Think of a database query result cache. The first time you run an expensive query, the database stores the result. Subsequent identical queries return the cached result instantly. KV cache does the same for attention: compute keys and values for each prompt token _once_ during prefill, write them to cache, and on every decode step only compute the one new token's K/V pair. Everything else is a cache lookup. The `past_key_values` parameter is literally the cache dictionary.


![KV cache mechanism: prefill processes all prompt tokens once, storing K/V; each decode step computes just 1 new K/V pair and appends it to the cache](images/kv-cache-mechanism.png)


In [ ]:
#  Part 1: KV-caching forward pass
class SimpleCausalAttention(nn.Module):
    """Minimal causal attention with optional KV cache."""

    def __init__(self, d_model=D_MODEL, n_heads=N_HEADS):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.scale = self.d_head**-0.5

    def forward(self, x, past_kv=None):
        """
        x:       (B, new_S, D)  — can be 1 token in decode mode
        past_kv: (k_cache, v_cache) each (B, past_S, n_heads, d_head), or None
        Returns: (output, (new_k_cache, new_v_cache))
        """
        B, new_S, D = x.shape
        # project to combined Q/K/V then split into heads
        qkv = self.W_qkv(x).reshape(B, new_S, 3, self.n_heads, self.d_head)
        q, k, v = qkv.unbind(dim=2)  # each (B, new_S, n_heads, d_head)

        # Extend KV cache
        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=1)  # (B, past_S+new_S, n_heads, d_head)
            v = torch.cat([past_kv[1], v], dim=1)

        # Scaled dot-product attention
        q = q.transpose(1, 2)  # (B, n_heads, new_S, d_head)
        k = k.transpose(1, 2)  # (B, n_heads, total_S, d_head)
        v = v.transpose(1, 2)

        scores = (
            torch.matmul(q, k.transpose(-2, -1)) * self.scale
        )  # (B, n_heads, new_S, total_S)
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, v)  # (B, n_heads, new_S, d_head)
        out = out.transpose(1, 2).reshape(B, new_S, D)
        return self.W_o(out), (k.transpose(1, 2), v.transpose(1, 2))


class ToyGPT(nn.Module):
    """Minimal GPT-style model with KV caching."""

    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, D_MODEL)
        # Stack N_LAYERS independent attention blocks
        self.layers = nn.ModuleList([SimpleCausalAttention() for _ in range(N_LAYERS)])
        self.norm = nn.LayerNorm(D_MODEL)
        self.head = nn.Linear(D_MODEL, VOCAB_SIZE, bias=False)

    def forward(self, x, past_kvs=None):
        """x: token ids (B, S). Returns (logits, new_past_kvs)."""
        h = self.embed(x)
        new_kvs = []
        # Run each layer with its own slice of the cached K/V state
        for i, layer in enumerate(self.layers):
            past = past_kvs[i] if past_kvs else None
            h, new_kv = layer(h, past)
            new_kvs.append(new_kv)
        return self.head(self.norm(h)), new_kvs


torch.manual_seed(42)
model = ToyGPT().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"ToyGPT: {n_params/1e3:.0f}K parameters (toy proxy for GPT-2)")

In [ ]:
#  Part 1: Compare with and without KV cache
PROMPT_LEN = 32  # prompt tokens
GEN_TOKENS = 20  # tokens to generate


def generate_no_cache(model, prompt_ids, n_tokens):
    """Standard generation: recompute all K/V at every step."""
    ids = prompt_ids.clone()
    t0 = time.perf_counter()
    # each step reprocesses the entire sequence so far — no cache reuse
    for _ in range(n_tokens):
        logits, _ = model(ids, past_kvs=None)  # recompute from scratch
        next_id = logits[:, -1, :].argmax(-1, keepdim=True)
        ids = torch.cat([ids, next_id], dim=1)
    return ids, time.perf_counter() - t0


def generate_with_cache(model, prompt_ids, n_tokens):
    """Cached generation: only process new token at each step."""
    # Prefill: process full prompt once, build KV cache
    _, past_kvs = model(prompt_ids, past_kvs=None)
    ids = prompt_ids.clone()
    t0 = time.perf_counter()
    # each step only feeds the newest token; past K/V is reused from cache
    for _ in range(n_tokens):
        logits, past_kvs = model(ids[:, -1:], past_kvs=past_kvs)  # 1 token!
        next_id = logits[:, 0, :].argmax(-1, keepdim=True)
        ids = torch.cat([ids, next_id], dim=1)
    return ids, time.perf_counter() - t0


prompt = torch.randint(0, VOCAB_SIZE, (1, PROMPT_LEN)).to(DEVICE)
model.eval()
# disable gradient tracking since we're only measuring inference speed
with torch.no_grad():
    # Warm up
    for _ in range(3):
        generate_no_cache(model, prompt, 5)
        generate_with_cache(model, prompt, 5)

    ids_no_cache, t_no_cache = generate_no_cache(model, prompt, GEN_TOKENS)
    ids_cached, t_cached = generate_with_cache(model, prompt, GEN_TOKENS)

# how many times faster the cached path is
speedup = t_no_cache / t_cached
# confirm caching doesn't change the generated tokens, only the speed
output_match = torch.equal(ids_no_cache, ids_cached)

print(f"Generating {GEN_TOKENS} tokens from {PROMPT_LEN}-token prompt:")
print(
    f"  Without KV cache: {t_no_cache*1000:.1f}ms  ({GEN_TOKENS/(t_no_cache):.0f} tok/s)"
)
print(
    f"  With KV cache:    {t_cached*1000:.1f}ms    ({GEN_TOKENS/(t_cached):.0f} tok/s)"
)
print(f"  Speedup: {speedup:.1f}×")
print(f"  Outputs identical: {output_match}")
print()
print(
    "Prediction check: answer (c) — each decode step computes 1 new K/V pair, reuses the rest."
)
print(
    f"  KV cache stores {N_LAYERS} layers × (K + V) × {PROMPT_LEN + GEN_TOKENS} positions"
)
kv_size_mb = N_LAYERS * 2 * (PROMPT_LEN + GEN_TOKENS) * D_MODEL * 4 / 1e6
print(f"  KV cache size: {kv_size_mb:.2f} MB (for this toy model)")
print(f"  For LLaMA-3-7B at S=2048: ~4 GB (manageable)")

#### What just happened — and what's missing

KV cache gave a measurable speedup by avoiding redundant attention computation. This is the most important single optimization for LLM inference.

**Missing piece:** KV cache solves per-token latency. But Riverside now has 100× more requests. Even with a fast single-request path, we can only serve one request at a time. We need to serve multiple requests **simultaneously** — that's continuous batching.


####  Your Turn — KV Cache: Prompt Length Impact

**Prediction:** If we double the prompt length from 32 to 64 tokens, the KV cache speedup will:

1. **(a) Increase** — more tokens means more redundant recomputation saved per decode step
2. **(b) Stay the same** — speedup only depends on the number of generated tokens
3. **(c) Decrease slightly** — longer prompts make the decode fraction proportionally smaller

Change `PROMPT_LEN_EXP` in the cell below and observe.


In [ ]:
#   Your Turn: Change prompt length
# # CHANGE: try PROMPT_LEN_EXP = 16, 64, 128 — how does speedup scale?
PROMPT_LEN_EXP = 32  # ← CHANGE ME

prompt_exp = torch.randint(0, VOCAB_SIZE, (1, PROMPT_LEN_EXP)).to(DEVICE)
model.eval()
# time both generation paths at this prompt length
with torch.no_grad():
    for _ in range(2):  # warm up
        generate_no_cache(model, prompt_exp, 5)
        generate_with_cache(model, prompt_exp, 5)
    _, t_nc = generate_no_cache(model, prompt_exp, GEN_TOKENS)
    _, t_c = generate_with_cache(model, prompt_exp, GEN_TOKENS)

# recompute speedup for the chosen prompt length
sp = t_nc / t_c
print(f"Prompt length: {PROMPT_LEN_EXP} tokens,  Generate: {GEN_TOKENS} tokens")
print(f"  Without KV cache: {t_nc*1000:.1f}ms")
print(f"  With KV cache:    {t_c*1000:.1f}ms")
print(f"  Speedup: {sp:.1f}×")
print()
print("Insight: longer prompts amplify KV cache savings because every decode step")
print("avoids recomputing attention over more previously-seen positions.")
print("Answer: (a) — longer prompts increase the speedup.")

---

## Part 2 — Continuous Batching: No Idle GPU Time

**Static batching** groups requests with the same length into a batch. Problem: short requests finish early and the GPU slot sits empty waiting for the longest request.

**Continuous batching** (vLLM, TGI): as soon as any request in the batch finishes, immediately slot in a new request. The GPU is always busy.

At 1000 req/min: static batching can waste 30–50% of GPU capacity in idle slots. Continuous batching reclaims that.


![Continuous batching eliminates idle GPU slots: static batching idles after short requests finish; continuous batching immediately slots in new requests](images/continuous-batching-vs-static.png)


#### #### Predict first — Continuous Batching Idle Time

Four requests arrive simultaneously at your gateway. They need [5, 8, 5, 6] decode steps respectively.  
With **static batching**, the entire batch waits until the _slowest_ request finishes (t = 8).

What fraction of total GPU slot-time is **wasted as idle** across all 4 slots?

1. **(a) < 5%** — small differences in length don't matter much
2. **(b) 15–25%** — noticeable waste but manageable overhead
3. **(c) > 30%** — a large fraction of GPU capacity thrown away even on this small example

Inspect the Gantt chart below to find out.


In [ ]:
#  Part 2: Static vs. Continuous Batching — Gantt-style GPU slot view
import matplotlib.patches as mpatches

# Two side-by-side panels: static batching (left) vs continuous batching (right)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

slot_colors = ["#2ecc71", "#3498db", "#e67e22", "#9b59b6"]

#  Static batching
# Batch 1: 4 requests, durations [5, 8, 5, 6]. All must wait until t=8 (the slowest).
# Batch 2 starts at t=8: durations [7, 4, 6, 7]. All wait until t=8+7=15.
ax1.set_title("Static Batching\n(wait for all to finish)", fontweight="bold")
batch1_durations = [5, 8, 5, 6]
batch1_end = max(batch1_durations)  # = 8  (whole batch blocked until slowest)
for slot, dur in enumerate(batch1_durations):
    ax1.barh(
        slot,
        dur,
        left=0,
        color=slot_colors[slot],
        alpha=0.85,
        height=0.6,
        label=f"Req {slot+1}",
    )
    idle = batch1_end - dur
    if idle > 0:
        ax1.barh(slot, idle, left=dur, color="#888", alpha=0.35, height=0.6)

batch2_durations = [7, 4, 6, 7]
batch2_end = batch1_end + max(batch2_durations)  # = 8 + 7 = 15
for slot, dur in enumerate(batch2_durations):
    ax1.barh(
        slot, dur, left=batch1_end, color=slot_colors[slot], alpha=0.55, height=0.6
    )
    idle = max(batch2_durations) - dur
    if idle > 0:
        ax1.barh(
            slot, idle, left=batch1_end + dur, color="#888", alpha=0.35, height=0.6
        )

ax1.set_xlabel("Time steps")
ax1.set_ylabel("GPU Slot")
ax1.set_yticks([0, 1, 2, 3])
ax1.set_yticklabels(["Slot 0", "Slot 1", "Slot 2", "Slot 3"])
ax1.set_xlim(0, 18)
ax1.axvline(batch1_end, color="#e74c3c", ls="--", lw=1.2, alpha=0.7)
ax1.axvline(batch2_end, color="#e74c3c", ls="--", lw=1.2, alpha=0.7)
ax1.text(
    9,
    -0.85,
    "GPU idle ~30% of time  (grey = wasted slots)",
    color="#c0392b",
    fontsize=8.5,
    ha="center",
)
grey_patch = mpatches.Patch(color="#888", alpha=0.4, label="Idle (wasted)")
ax1.legend(handles=[grey_patch], loc="upper right", fontsize=8)

#  Continuous batching
# As soon as a slot finishes its request, a new request fills it immediately.
# Slot 0: t=0–5 (reqA), t=5–11 (reqE),  t=11–17 (reqI)
# Slot 1: t=0–8 (reqB), t=8–13  (reqF), t=13–17 (reqJ)
# Slot 2: t=0–5 (reqC), t=5–12  (reqG), t=12–17 (reqK)
# Slot 3: t=0–6 (reqD), t=6–13  (reqH), t=13–17 (reqL)
ax2.set_title(
    "Continuous Batching\n(fill slot immediately on finish)", fontweight="bold"
)
# hard-coded slot timelines for the continuous-batching illustration (3 requests back-to-back per slot)
continuous_slots = [
    [(0, 5), (5, 11), (11, 17)],  # slot 0
    [(0, 8), (8, 13), (13, 17)],  # slot 1
    [(0, 5), (5, 12), (12, 17)],  # slot 2
    [(0, 6), (6, 13), (13, 17)],  # slot 3
]
for slot, segs in enumerate(continuous_slots):
    for i, (start, end) in enumerate(segs):
        ax2.barh(
            slot,
            end - start,
            left=start,
            color=slot_colors[slot],
            alpha=0.85 - i * 0.12,
            height=0.6,
        )

ax2.set_xlabel("Time steps")
ax2.set_ylabel("GPU Slot")
ax2.set_yticks([0, 1, 2, 3])
ax2.set_yticklabels(["Slot 0", "Slot 1", "Slot 2", "Slot 3"])
ax2.set_xlim(0, 18)
ax2.text(
    9,
    -0.85,
    "GPU idle ~0%  — slots fill immediately, no grey patches",
    color="#27ae60",
    fontsize=8.5,
    ha="center",
)

plt.suptitle(
    "Static vs. Continuous Batching: GPU Utilisation", fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.show()

print(
    "\n→ Continuous batching: 12 requests served in 17 time-steps vs. 15 steps for 8 requests static (50% more requests, same wall-clock time)."
)
print(
    "→ Riverside handles a 100× traffic spike by eliminating idle slots — not by adding hardware."
)

#### What just happened — quantifying the idle-slot elimination

The Gantt chart above simulates the same 4 GPU slots serving requests two ways:

- **Static batching:** the grey segments are idle slot-time — a slot sits empty waiting for the slowest request in its batch. Summed across both batches: 12 idle slot-time-units out of 60 total (4 slots × 15 time-steps) = **20% of GPU capacity wasted**.
- **Continuous batching:** no grey segments anywhere — **0% idle** across 68 slot-time-units (4 slots × 17 time-steps). The instant a request finishes, a new one takes its slot.

That's a **~100% reduction in wasted GPU-slot-time** (20% → 0%), and it's why continuous batching served 12 requests — 50% more — in roughly the same wall-clock window that static batching used for only 8. For Riverside's 100× traffic spike, reclaiming that idle capacity — not buying more GPUs — is what makes the spike survivable.


#### What just happened — and what's missing

Continuous batching filled the GPU's idle slots. But 100 concurrent users each with a KV cache pre-allocated at max sequence length means we're reserving 2048 positions per user even if they only used 50 — 97.5% of that KV cache memory is wasted. **Next: PagedAttention reclaims that fragmented memory.**


####  Your Turn — Continuous Batching: Slot Count Impact

**Prediction:** If we double from 4 GPU slots to 8, idle percentage with static batching will:

1. **(a) Decrease** — more slots means shorter batches, less wait time
2. **(b) Increase** — a wider pool has a larger gap between fastest and slowest request
3. **(c) Stay the same** — idle fraction is independent of slot count

Change `n_slots_exp` below and observe.


In [ ]:
#   Your Turn: Change number of GPU slots
# # CHANGE: try n_slots_exp = 2, 8, 16 — how does idle fraction change?
n_slots_exp = 4  # ← CHANGE ME

np.random.seed(42)
# simulate a random decode length for each of the n_slots_exp concurrent requests
request_durations = np.random.randint(3, 12, size=n_slots_exp).tolist()
max_dur = max(request_durations)
# total wasted slot-time: every slot idles until the slowest request finishes
idle_time = sum(max_dur - d for d in request_durations)
total_time = max_dur * n_slots_exp
# fraction of total GPU slot-time that went unused
idle_fraction = idle_time / total_time

print(f"Slots: {n_slots_exp},  Request durations: {request_durations}")
print(f"  Batch ends at: t={max_dur}  (slowest request)")
print(f"  Idle slot-time: {idle_time} / {total_time} = {idle_fraction:.1%}")
print()
print("Insight: idle fraction tends to grow with more slots because a wider pool")
print("creates a larger gap between the fastest and slowest request in the batch.")
print("Answer: (b) — more slots generally worsens idle fraction with static batching.")

---

## Part 3 — PagedAttention: Eliminating KV Cache Fragmentation

**The problem:** Traditional KV cache pre-allocates contiguous memory for max sequence length. If you reserve 2048 tokens for every request but average request is 200 tokens: 90% of the allocated memory is wasted.

**PagedAttention** (from vLLM): manage KV cache like an OS manages virtual memory — in fixed-size **pages** of 16 or 32 tokens. Only allocate pages as tokens are generated. A page table maps logical positions to physical memory blocks.

Result: ~90% GPU memory utilization vs. ~30–50% with pre-allocation.


#### #### Predict first — PagedAttention Memory Utilization

Your inference server pre-allocates KV cache for **512 tokens** per request (max sequence length).  
Real requests follow an exponential distribution — most are short (~100 tokens avg), a few are long.

What fraction of the pre-allocated KV cache memory is **actually used**?

1. **(a) 60–80%** — short requests still use most of their allocation
2. **(b) 30–50%** — moderate waste due to the long-tail distribution
3. **(c) < 20%** — the exponential distribution means most allocated memory sits empty

Run the cell below to measure the real utilization.


In [ ]:
#  Part 3: PagedAttention memory utilization illustration
PAGE_SIZE = 16  # tokens per page


def preallocated_utilization(requests, max_seq_len=512):
    """Old approach: reserve max_seq_len per request."""
    total_reserved = len(requests) * max_seq_len
    actually_used = sum(requests)
    return actually_used / total_reserved


def paged_utilization(requests, page_size=PAGE_SIZE):
    """PagedAttention: allocate only the pages actually needed."""
    pages_needed = sum(np.ceil(r / page_size) for r in requests)
    tokens_in_pages = pages_needed * page_size
    actually_used = sum(requests)
    return actually_used / tokens_in_pages


# Simulate realistic request distribution
np.random.seed(42)
request_lens_dist = (
    np.random.exponential(scale=100, size=100).clip(10, 512).astype(int).tolist()
)

# compare the two allocation strategies on the same simulated request distribution
util_preallocated = preallocated_utilization(request_lens_dist)
util_paged = paged_utilization(request_lens_dist)

print(f"KV cache memory utilization ({len(request_lens_dist)} concurrent requests):")
print(f"  Pre-allocated (max={512}):  {util_preallocated:.1%}")
print(f"  PagedAttention (page={PAGE_SIZE}): {util_paged:.1%}")
print()
print(f"  PagedAttention improves utilization by {util_paged/util_preallocated:.1f}×")
print(
    f"  This means {util_paged/util_preallocated:.1f}× more concurrent requests on the same GPU"
)
print()

# Distribution plot
# visualize the request-length distribution and how much of it exceeds typical usage
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(request_lens_dist, bins=30, color="steelblue", alpha=0.7)
ax.axvline(
    np.mean(request_lens_dist),
    color="coral",
    ls="--",
    lw=2,
    label=f"Mean: {np.mean(request_lens_dist):.0f} tokens",
)
ax.axvline(512, color="gray", ls=":", lw=1, label="Max (pre-allocation waste)")
ax.set_xlabel("Request length (tokens)")
ax.set_ylabel("Count")
ax.set_title(
    "Request length distribution — exponential tail creates pre-allocation waste"
)
ax.legend()
plt.tight_layout()
plt.show()

#### What just happened — and what's missing

PagedAttention eliminated KV cache fragmentation, allowing 3× more concurrent users on the same GPU. But each individual request still takes the same time — decode is serial. TTFT for a 200-token response is still 200 × decode_latency. **Next: speculative decoding trades a small draft model's speed for the large model's quality, getting multiple tokens per verifier call.**


####  Your Turn — PagedAttention: Page Size Trade-off

**Prediction:** If we shrink the page size from 16 to 4 tokens, memory utilization will:

1. **(a) Improve** — smaller pages = finer granularity = less wasted space per request
2. **(b) Worsen** — smaller pages increase page-table overhead, hurting effective utilization
3. **(c) Stay the same** — page size doesn't affect utilization at scale

Change `PAGE_SIZE_EXP` below to explore the trade-off.


In [ ]:
#   Your Turn: Change page size
# # CHANGE: try PAGE_SIZE_EXP = 4, 8, 32, 64 — how does utilization change?
PAGE_SIZE_EXP = 16  # ← CHANGE ME

np.random.seed(42)
req_lens_exp = (
    np.random.exponential(scale=100, size=100).clip(10, 512).astype(int).tolist()
)

# how many fixed-size pages each request rounds up to at this page size
pages_needed = sum(int(np.ceil(r / PAGE_SIZE_EXP)) for r in req_lens_exp)
tokens_in_pages = pages_needed * PAGE_SIZE_EXP
actually_used = sum(req_lens_exp)
# fraction of allocated (paged) memory actually holding real tokens
util = actually_used / tokens_in_pages

print(f"Page size: {PAGE_SIZE_EXP} tokens")
print(f"  Actually used tokens:     {actually_used:,}")
print(f"  Tokens allocated (pages): {int(tokens_in_pages):,}")
print(f"  Memory utilization:       {util:.1%}")
print()
# Show the trade-off across page sizes for comparison
for ps in [4, 8, 16, 32, 64]:
    u = paged_utilization(req_lens_exp, ps)
    print(f"  page_size={ps:3d}: {u:.1%} utilization")
print()
print(
    "Insight: smaller pages reduce waste at allocation boundaries (better utilization)"
)
print("but extremely small pages add page-table bookkeeping overhead in real kernels.")
print(
    "Answer: (a) — smaller pages improve utilization up to a practical minimum (~8–16)."
)

---

## Part 4 — Speculative Decoding: Parallelise Token Verification

**Problem:** LLM decode is serial — token N must be generated before token N+1.

**Speculative decoding:** use a small, fast **draft model** to propose K tokens at once. Then run the large **verifier** model on all K tokens in a single parallel forward pass.

- If the verifier agrees with all K draft tokens: accept all K tokens at once
- If it disagrees at position j: accept tokens 1..j-1, reject the rest, and sample a corrected token j from the verifier

Net result: most of the time, 3–5 tokens are accepted per verifier call instead of 1.

#### #### Predict first

A 7B verifier and a 70M draft model. The draft proposes 5 tokens; verifier accepts 3. Compared to 5 sequential 7B model calls:

1. **(a) 2× faster** — draft model speed + 3 tokens per verifier call
2. **(b) 5× faster** — equivalent to 5 sequential tokens in 1 verifier call
3. **(c) Marginally faster** — the draft model overhead negates most savings


> **Why parallel verification works:** A transformer is inherently parallel — given N input tokens, it produces N output logits in one forward pass. Autoregressive generation artificially enforces serial order (token N must exist before generating N+1). Speculative decoding breaks that constraint: the small draft model proposes K tokens sequentially (it's fast). The large verifier then does what transformers naturally do — processes all K positions simultaneously in one parallel pass and checks which tokens it agrees with. If the draft was right, you get K tokens for the price of one large-model call.


![Speculative decoding: draft model proposes 5 tokens, verifier accepts 3 (teal checks) and rejects 2 (coral crosses) in one parallel forward pass](images/speculative-decoding-accept-reject.png)


In [ ]:
#  Part 4: Speculative decoding simulation
def simulate_speculative_decoding(
    n_tokens_to_generate=50,
    draft_tokens_per_step=4,
    acceptance_rate=0.75,  # fraction of draft tokens accepted
    draft_latency_ms=1.0,  # small model
    verifier_latency_ms=10.0,  # large model
):
    """Simulate speculative decoding and compare to sequential generation."""

    # Sequential baseline: one verifier call per token
    t_sequential = n_tokens_to_generate * verifier_latency_ms

    # Speculative: draft K tokens, then verify once
    tokens_generated = 0
    t_speculative = 0
    n_verifier_calls = 0

    # keep drafting+verifying rounds until the target token count is reached
    while tokens_generated < n_tokens_to_generate:
        # Draft K tokens
        t_speculative += draft_tokens_per_step * draft_latency_ms
        # Verifier forward pass (parallel)
        t_speculative += verifier_latency_ms
        n_verifier_calls += 1
        # Accept according to acceptance rate
        accepted = (
            int(draft_tokens_per_step * acceptance_rate) + 1
        )  # +1 for verifier's own
        tokens_generated += min(accepted, n_tokens_to_generate - tokens_generated)

    # how many times faster speculative decoding is at this configuration
    speedup = t_sequential / t_speculative
    return t_sequential, t_speculative, speedup, n_verifier_calls


t_seq, t_spec, speedup, n_calls = simulate_speculative_decoding()
print(f"Speculative decoding simulation ({50} tokens, draft_len=4, accept_rate=75%):")
print(f"  Sequential (7B model):     {t_seq:.0f}ms  →  {50/t_seq*1000:.0f} tok/s")
print(f"  Speculative (7B+70M):      {t_spec:.0f}ms  →  {50/t_spec*1000:.0f} tok/s")
print(f"  Speedup: {speedup:.1f}×  ({n_calls} verifier calls instead of 50)")
print()
print("Prediction check:")
# check whether the measured speedup matches the predicted outcome
if speedup >= 2.0:
    print(
        f"  Answer (a) confirmed: {speedup:.1f}× speedup with 3-4 tokens per verifier call"
    )
else:
    print(
        f"  Answer (c) — small speedup at this acceptance rate; optimal at rate > 80%"
    )
print()

# Sweep acceptance rate
# sweep acceptance rate to see how speedup depends on draft-model quality
rates = np.linspace(0.5, 1.0, 11)
# recompute speedup at each acceptance rate in the sweep
speedups = [simulate_speculative_decoding(acceptance_rate=r)[2] for r in rates]

# plot speedup vs. acceptance rate to find the break-even point
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rates * 100, speedups, "o-", color="steelblue", lw=2)
ax.axhline(1.0, color="gray", ls="--", lw=1, label="No speedup baseline")
ax.set_xlabel("Token acceptance rate (%)")
ax.set_ylabel("Speedup (×)")
ax.set_title("Speculative decoding speedup vs. draft model acceptance rate")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print("→ Acceptance rate > 70% required for meaningful speedup.")
print("  Small models fine-tuned on the target domain achieve 80-90% acceptance.")

---

##  Your Turn — Speculative Decoding Draft Length

**Prediction:** If we increase draft length from 4 to 8 tokens per step (more speculation), will the mean acceptance rate:

1. Go up — more speculation = more aggressive = higher throughput
2. Go down — longer chains are harder to accept wholesale
3. Stay the same — acceptance rate doesn't depend on draft length

Change `draft_tokens_per_step` to 8 in the cell below and observe.


In [ ]:
#   Your Turn: Change draft length
# # CHANGE: try draft_tokens_per_step = 8, 12 — does speedup improve?
draft_len = 4  # ← CHANGE ME

# compare speedup at this draft length across a range of acceptance rates
for rate in [0.6, 0.75, 0.9]:
    _, _, speedup, _ = simulate_speculative_decoding(
        draft_tokens_per_step=draft_len, acceptance_rate=rate
    )
    print(f"  draft_len={draft_len}, accept_rate={rate:.0%}: {speedup:.1f}× speedup")

print()
print("Observation: longer draft chains increase throughput when acceptance is high,")
print("but add overhead when acceptance is low — the optimal draft length depends on")
print("the domain-specific acceptance rate of the draft model.")

#### What just happened — and what's missing

Speculative decoding showed that a small 70M draft model proposing 4 tokens at a time — with 75% acceptance — delivers roughly **2× speedup** over purely sequential 7B-model generation. The key insight: the large model's native parallelism (it processes N tokens in one forward pass) was being wasted by autoregressive serial decoding; speculative decoding harvests that parallelism.

**Missing piece:** We've optimised individual-request latency (KV cache, speculative decoding) and batch efficiency (continuous batching, PagedAttention). But we haven't asked: does the GPU spend its time _differently_ during prompt processing vs. token generation? That asymmetry determines how to size and schedule a production cluster — and it's the subject of Part 5.


---

## Part 5 — Prefill vs. Decode: Different Phases, Different Optimal Batch Sizes

> **Riverside at 100× load:** Prefill (processing the prompt) and decode (generating tokens one by one) have very different GPU profiles. At high traffic, separating these onto different hardware can double throughput.

**Prefill:** process all prompt tokens in one parallel forward pass. High arithmetic intensity (large matmuls). Compute-bound. Benefits from large batch size.

**Decode:** generate one token at a time. Low arithmetic intensity (1-token inputs × full KV cache). Memory-bound. Benefits from large batch size differently (amortize weight reads).

This asymmetry is why production servers disaggregate prefill and decode onto separate GPU instances (disaggregated serving).


#### #### Predict first — Prefill vs. Decode Timing

Our toy model processes a 256-token prompt (prefill: all tokens in parallel) vs. a single decode step (one token with KV cache).

How much **longer** does the 256-token prefill take compared to a single decode step?

1. **(a) ~2–3× longer** — slightly more work but the same transformer architecture
2. **(b) ~10–20× longer** — processes all 256 tokens at once vs. 1 token
3. **(c) About the same** — both paths go through the same number of layer operations

Run the timing cell below to find out.


In [ ]:
#  Part 5: Prefill vs. decode phase comparison
PROMPT_LENGTHS = [32, 64, 128, 256]

prefill_times_ms = []
decode_times_ms = []

model.eval()
with torch.no_grad():
    # measure prefill and decode latency separately at each prompt length
    for plen in PROMPT_LENGTHS:
        # Prefill: process all tokens at once
        prompt_ids = torch.randint(0, VOCAB_SIZE, (1, plen)).to(DEVICE)
        if HAS_GPU:
            torch.cuda.synchronize()
        times_p = []
        # repeat and take the median to reduce timing noise
        for _ in range(10):
            if HAS_GPU:
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            _, past_kvs = model(prompt_ids)
            if HAS_GPU:
                torch.cuda.synchronize()
            times_p.append((time.perf_counter() - t0) * 1000)
        prefill_times_ms.append(np.median(times_p))

        # Decode: one token at a time with KV cache
        times_d = []
        # repeat the single-token decode step to get a stable median timing
        for _ in range(10):
            if HAS_GPU:
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            next_id = torch.randint(0, VOCAB_SIZE, (1, 1)).to(DEVICE)
            _, _ = model(next_id, past_kvs=past_kvs)
            if HAS_GPU:
                torch.cuda.synchronize()
            times_d.append((time.perf_counter() - t0) * 1000)
        decode_times_ms.append(np.median(times_d))

print(f"Prefill vs. decode timing (toy model):")
print(
    f"{'Prompt len':12s}  {'Prefill (ms)':14s}  {'Decode/tok (ms)':16s}  {'Prefill/Decode':14s}"
)
print("-" * 60)
# print the prefill/decode ratio for each prompt length
for plen, tp, td in zip(PROMPT_LENGTHS, prefill_times_ms, decode_times_ms):
    print(f"  {plen:10d}   {tp:10.2f}      {td:12.2f}           {tp/td:8.1f}×")

print()
print(
    "Key insight: prefill (prompt processing) takes proportionally more time than decode"
)
print("because it processes S tokens in parallel — more FLOP per step.")
print("Decode is faster per token but limited by memory bandwidth (KV cache reads).")
print()
print("→ Production systems often use separate GPU pools: prefill (compute-intensive)")
print(
    "  and decode (memory-intensive) with different hardware and batching strategies."
)

####  Your Turn — Prefill vs. Decode: Batch Size Effect

**Prediction:** If we run decode with a batch of 8 requests instead of 1, the **per-token** decode time will:

1. **(a) Increase ~8×** — 8 tokens to process per step means 8× the work
2. **(b) Stay almost the same** — transformer weight reads are amortised over the whole batch
3. **(c) Decrease per-token** — larger batches improve GPU utilisation for memory-bound work

Change `DECODE_BATCH` below and observe.


In [ ]:
#   Your Turn: Batch size effect on decode latency
# # CHANGE: try DECODE_BATCH = 1, 4, 8, 16 — does per-token time change?
DECODE_BATCH = 1  # ← CHANGE ME

model.eval()
with torch.no_grad():
    # Build a shared KV cache for the batch
    prompt_batch = torch.randint(0, VOCAB_SIZE, (DECODE_BATCH, 64)).to(DEVICE)
    _, past_kvs_batch = model(prompt_batch)

    times_batch = []
    # time the decode step repeatedly at this batch size
    for _ in range(20):
        next_ids = torch.randint(0, VOCAB_SIZE, (DECODE_BATCH, 1)).to(DEVICE)
        if HAS_GPU:
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        _, _ = model(next_ids, past_kvs=past_kvs_batch)
        if HAS_GPU:
            torch.cuda.synchronize()
        times_batch.append((time.perf_counter() - t0) * 1000)

# normalize total step time by batch size to get per-token cost
per_tok = np.median(times_batch) / DECODE_BATCH
print(f"Decode batch size: {DECODE_BATCH}")
print(f"  Total step time:   {np.median(times_batch):.2f}ms")
print(f"  Per-token time:    {per_tok:.3f}ms/tok")
print()
print(
    "Insight: decode is memory-bandwidth-bound (reading KV cache + weights per step)."
)
print("Larger batches amortise the fixed weight-read cost across more tokens,")
print("reducing per-token time — until KV cache reads themselves become the limit.")
print("Answer: (b)/(c) — per-token time decreases or stays flat as batch grows.")

#### What just happened — and what's missing

The timing data confirms that prefill (processing N tokens in parallel) is **~10–20× slower per-step than decode** — not because it does different work, but because it does proportionally more of it (N tokens of matmuls vs. 1). This asymmetry is why production systems like vLLM 0.5+ disaggregate prefill and decode onto separate GPU pools: a prefill-optimised GPU (high compute throughput) and a decode-optimised GPU (high memory bandwidth) each do what they're best at.

**Notebook complete:** Parts 1–5 covered the five key software-level optimisations. Part 6 maps these to the production metrics — TTFT and TPOT — that a real team would monitor in vLLM.


---

## Part 6 — Toy → Real: Mapping Optimizations to Production Metrics

Each technique from Parts 1–5 maps to a measurable metric in production serving. The table below bridges our toy numbers to what you'd observe in vLLM, TGI, or TensorRT-LLM.

| Technique                    | Toy result (this notebook)                  | Production impact                               | Key metric                       | vLLM / TensorRT flag               |
| ---------------------------- | ------------------------------------------- | ----------------------------------------------- | -------------------------------- | ---------------------------------- |
| KV cache                     | 2–5× speedup on toy (32-token prompt)       | 3–10× TPOT reduction on LLaMA-3-7B at S=2048    | **TPOT** (time per output token) | Always-on; `past_key_values` in HF |
| Continuous batching          | 50% more requests in same wall-clock window | 2–4× req/s improvement at real load             | **Throughput** (req/s)           | Default in vLLM; `--max-num-seqs`  |
| PagedAttention               | 3–5× memory utilisation improvement (toy)   | ~3× more concurrent users on A100 80GB          | **GPU mem util %**               | vLLM core; `--block-size 16`       |
| Speculative decoding         | 2× speedup at 75% acceptance rate (toy sim) | 1.5–3× TPOT + TTFT on domain-matched model      | **TTFT + TPOT**                  | `--speculative-model <draft>`      |
| Disaggregated prefill/decode | 10–20× prefill/decode time ratio (toy)      | Separate TTFT (prefill-GPU) / TPOT (decode-GPU) | **TTFT separately**              | PD disaggregation in vLLM 0.5+     |

> **Plain-English gloss:** TTFT (Time To First Token) is what the user _feels_ as "how long until anything appears." TPOT (Time Per Output Token) is the streaming speed after that. A slow TTFT feels like a lag; a slow TPOT feels like sluggish typing. The optimizations above target different halves of the user experience — and they don't all help the same metric.

### How the improvements compound — from 10 to 1,000 req/min

| Stage                   | Optimization added                               | Multiplier | Cumulative req/min |
| ----------------------- | ------------------------------------------------ | ---------- | ------------------ |
| Baseline                | Naive sequential generation                      | 1×         | 10                 |
| + KV cache              | Eliminate redundant attention recomputation      | ~4×        | 40                 |
| + Continuous batching   | Fill idle GPU slots between requests             | ~2.5×      | 100                |
| + PagedAttention        | Serve 3× more concurrent users from same GPU RAM | ~3×        | 300                |
| + Speculative decoding  | 2× faster generation per request                 | ~2×        | 600                |
| + Disaggregated serving | Optimise TTFT and TPOT independently             | ~1.7×      | ~1,000             |

Each row compounds on the previous — all five techniques together, and software alone takes you from 10 to ~1,000 req/min. No new hardware required.


In [ ]:
#  Closing Decision
# Compute cumulative throughput improvement using results from each Part
baseline_rps = 10.0  # requests/min
kv_mult = speedup  # from Part 1 benchmark
cb_mult = 1.5  # from Part 2 simulation: 12 requests vs 8 in same window
sd_mult = simulate_speculative_decoding()[2]  # from Part 4

# compound each optimization's multiplier onto the running requests/min estimate
kv_rps = baseline_rps * kv_mult
cb_rps = kv_rps * cb_mult
sd_rps = cb_rps * sd_mult
target = 1000.0

print("=" * 60)
print("  CLOSING DECISION — Riverside 100× Traffic Scaling")
print("=" * 60)
print()
print(f"  Starting point: {baseline_rps:.0f} req/min (naive serving)")
print()
print(f"  + KV cache:            {kv_rps:.0f} req/min  ({kv_mult:.1f}× from Part 1)")
print(f"  + Continuous batching: {cb_rps:.0f} req/min  ({cb_mult:.1f}× from Part 2)")
print(f"  + Speculative decoding:{sd_rps:.0f} req/min  ({sd_mult:.1f}× from Part 4)")
print()
print(f"  Target: {target:.0f} req/min")
# check whether the compounded improvements hit Riverside's 1,000 req/min target
achieved = sd_rps >= target
print(
    f"  Achieved with software alone: {' YES' if achieved else ' Not quite — PagedAttention + disaggregation close the gap'}"
)
print()
print("  Production recommendation:")
print(
    "  1. Deploy via vLLM (KV cache + continuous batching + PagedAttention out-of-box)"
)
print("  2. Enable speculative decoding with a domain-fine-tuned 70M draft model")
print("  3. Monitor TTFT (user-perceived latency) and TPOT (throughput) separately")
print()
print("  One-line vLLM deployment:")
print(
    "    vllm serve <model-name> --enable-chunked-prefill --speculative-model <draft>"
)

---

## What This Notebook Covered (and What It Didn't)

### Tier 1 — Implemented and Demonstrated

- KV cache — real PyTorch forward pass; verified outputs match non-cached; speedup measured
- Continuous batching — simulated; throughput comparison vs. static batching
- PagedAttention — memory utilization modeled; fragmentation vs. paged compared
- Speculative decoding — simulated; acceptance rate sweep; speedup quantified
- Prefill vs. decode — timing comparison across prompt lengths

### Tier 2 — Explained but Not Fully Built

- **PagedAttention** — the page table mechanism is modeled numerically; a real block-level attention implementation requires modifying the attention kernel

### Tier 3 — Named but Out of Scope

- **Tensor-parallel inference** — split the model across GPUs for low-latency high-throughput serving; covered in Ch5 (Distributed)
- **Disaggregated prefill-decode** — separate GPU pools for each phase; used at scale in production clusters
- **Chunked prefill** — interleave long-prompt prefill with decode to reduce TTFT jitter


---

## When to Use What

| Optimization          | Benefit                   | When to add                                   |
| --------------------- | ------------------------- | --------------------------------------------- |
| KV cache              | 3–10× faster decode       | Always — built into all frameworks            |
| Continuous batching   | 2–3× throughput           | Any serving scenario with variable lengths    |
| PagedAttention        | ~3× more concurrent users | When GPU memory is the constraint             |
| Speculative decoding  | 1.5–3× faster generation  | When you have a good draft model (fine-tuned) |
| Disaggregated serving | Optimise TTFT separately  | At >100 req/s with mixed short/long prompts   |

→ **Next:** `learning/ai-infrastructure/08-triton-kernels/` — all the optimizations above depend on fast GPU kernels underneath. This chapter shows how to write them in Python using Triton.


---

## Summary — What This Notebook Built

| Part | Optimization         | Proved with                        | Key number                              |
| ---- | -------------------- | ---------------------------------- | --------------------------------------- |
| 1    | KV cache             | PyTorch forward-pass benchmark     | 2–5× speedup (toy); 3–10× in production |
| 2    | Continuous batching  | Gantt-chart simulation             | 0% idle → 50% more requests per window  |
| 3    | PagedAttention       | Memory utilisation model           | 3–5× utilisation improvement            |
| 4    | Speculative decoding | Acceptance-rate sweep              | 2× speedup at 75% acceptance            |
| 5    | Prefill vs. decode   | Timing across prompt lengths       | 10–20× prefill/decode time ratio        |
| 6    | Toy → real bridge    | Metric mapping + compounding table | 10 → ~1,000 req/min with software only  |

### Key insights to keep

- **KV cache is the highest-leverage single change** — it eliminates O(S²) redundant attention work in every decode step. It's on by default in every production framework; understanding _why_ prevents accidentally disabling it during a "speed optimisation."
- **Continuous batching and PagedAttention are multiplicative** — batching fills idle GPU slots, paging fills idle GPU memory. Together they push utilisation from ~30% to ~95% without touching the model.
- **Speculative decoding only wins when the draft model is domain-matched** — a generic 70M model on medical text achieves 40–50% acceptance (marginal gain); one fine-tuned on the domain achieves 80–90% (2–3× speedup). The difference between "barely worth it" and "transformative."
- **Prefill is compute-bound; decode is memory-bound** — this is a fundamental property, not a tunable parameter. Disaggregated serving (separate GPU pools for each phase) is the production answer, not over-provisioning one.
- **All five optimisations compound** — KV cache × continuous batching × PagedAttention × speculative decoding × disaggregation gives roughly 40–100× improvement over naive serving. The bottleneck after model training is rarely the model — it's the serving stack around it.


---

## Part 7 - Production and Cloud Inference Deployment

The kernels and schedulers from Parts 1-6 become a reliable service only when the deployment protects latency, memory, and correctness under changing traffic. Treat the serving runtime as a bounded system: every request consumes prompt tokens, decode slots, KV-cache pages, and time from a finite latency budget.

### From model artifact to ready replica

1. **Load immutably.** Pin the model revision, tokenizer revision, precision, tensor-parallel layout, and serving-engine version. Download or mount artifacts before accepting traffic; never resolve a floating model name in the request path.
2. **Warm deliberately.** Allocate model weights and KV-cache blocks, compile kernels if required, and run representative local warm-up shapes. A process can be alive while still unable to meet TTFT, so liveness and readiness must be separate.
3. **Expose readiness last.** Mark a replica ready only after artifact verification, model load, cache allocation, warm-up, and a local inference probe succeed. Remove readiness before draining or reloading.

### Scheduling, memory, and overload policy

| Concern | Production policy | Connection to this notebook |
| --- | --- | --- |
| Batching | Use continuous batching with bounded waiting time and token-aware batch limits | Part 2 fills idle slots without making short requests wait behind long ones |
| KV cache | Budget cache by active prompt plus decode tokens; page allocations and reject before OOM | Parts 1 and 3 make cache reuse and fragmentation visible |
| Concurrency | Bound admitted sequences and queued tokens, not just HTTP connections | One streaming request can hold a decode slot and KV pages for seconds |
| Backpressure | Return an explicit retryable overload response before the queue becomes unbounded | A short, controlled rejection is better than a timeout storm |
| Deadlines | Carry an absolute deadline through queue, prefill, and decode; stop work after expiry | Queue time consumes the same user-visible SLO as TTFT and TPOT |
| Long prompts | Cap input and output tokens or route long-context work to a separate pool | Prefill is compute-bound while decode is memory-bandwidth-bound |

### Cloud deployment loop

- **Autoscale on saturation, not CPU alone.** Useful signals include admitted-token rate, queue delay, active sequences, KV-cache utilization, TTFT, and tokens per second. Scale prefill and decode pools independently when they are disaggregated.
- **Route by capability.** Select pools by model revision, context length, region, tenant policy, and latency class. Keep fallback explicit: retry only idempotent requests, preserve the remaining deadline, and avoid silently switching to a model with different quality or safety behavior.
- **Define SLOs from the user experience.** Track availability, p50/p95/p99 TTFT, p95/p99 TPOT, end-to-end latency, and deadline success by model and route. Error budgets decide whether to ship optimizations or stabilize the service.
- **Account for cost per useful work.** Monitor GPU-hours, idle capacity, input/output tokens, cache hit and eviction rates, rejected work, and cost per successful million tokens. Higher utilization is not a win if tail latency or failure rate breaks the SLO.
- **Instrument every phase.** Emit request IDs and traces spanning admission, queue, prefill, first token, decode, and completion. Metrics must be bounded-cardinality; logs must redact prompts and generated text unless an approved sampling policy says otherwise.
- **Release with evidence.** Canary a pinned model/runtime/config tuple to a small traffic slice, compare quality and serving SLOs, then increase gradually. Automatic rollback should restore the entire tuple when error rate, TTFT, TPOT, memory pressure, or quality guardrails regress.

### An immutable serving contract

Production configuration should describe one deployable unit and be replaced atomically, not mutated request by request. The example below pins the artifact and runtime, makes batching and KV-cache capacity explicit, and keeps every potentially production-facing action disabled. It uses only the Python standard library and does not start a server or contact a cloud service.

In [ ]:
from dataclasses import dataclass
from typing import Tuple

# Safety gates: these remain False in the checked-in notebook.
RUN_PRODUCTION_SMOKE = False
RUN_PRODUCTION_LOAD = False
RUN_PRODUCTION_CANARY = False


@dataclass(frozen=True)
class ModelArtifact:
    model_id: str
    model_revision: str
    tokenizer_revision: str
    engine_version: str
    dtype: str


@dataclass(frozen=True)
class CapacityLimits:
    max_model_tokens: int
    max_input_tokens: int
    max_output_tokens: int
    max_active_sequences: int
    max_batch_tokens: int
    kv_cache_budget_bytes: int
    kv_cache_reserve_fraction: float


@dataclass(frozen=True)
class ServingSLO:
    request_deadline_ms: int
    p95_ttft_ms: int
    p95_tpot_ms: int
    availability_target: float


@dataclass(frozen=True)
class ServingConfig:
    artifact: ModelArtifact
    capacity: CapacityLimits
    slo: ServingSLO
    regions: Tuple[str, ...]
    fallback_model_id: str | None


SERVING_CONFIG = ServingConfig(
    artifact=ModelArtifact(
        model_id="riverside-editor-7b",
        model_revision="sha256:replace-with-verified-model-digest",
        tokenizer_revision="sha256:replace-with-verified-tokenizer-digest",
        engine_version="pinned-serving-engine-version",
        dtype="bfloat16",
    ),
    capacity=CapacityLimits(
        max_model_tokens=4096,
        max_input_tokens=3072,
        max_output_tokens=1024,
        max_active_sequences=64,
        max_batch_tokens=8192,
        kv_cache_budget_bytes=20 * 1024**3,
        kv_cache_reserve_fraction=0.10,
    ),
    slo=ServingSLO(
        request_deadline_ms=10_000,
        p95_ttft_ms=800,
        p95_tpot_ms=60,
        availability_target=0.999,
    ),
    regions=("primary-region", "secondary-region"),
    fallback_model_id=None,  # Fallback requires an explicit quality/safety review.
)

print("Pinned model:", SERVING_CONFIG.artifact.model_id)
print("Production smoke enabled:", RUN_PRODUCTION_SMOKE)
print("Production load enabled:", RUN_PRODUCTION_LOAD)
print("Production canary enabled:", RUN_PRODUCTION_CANARY)

In [ ]:
@dataclass(frozen=True)
class RequestPlan:
    request_id: str
    input_tokens: int
    max_new_tokens: int
    remaining_deadline_ms: int


@dataclass(frozen=True)
class CapacitySnapshot:
    active_sequences: int
    scheduled_batch_tokens: int
    kv_cache_bytes_used: int


@dataclass(frozen=True)
class AdmissionDecision:
    accepted: bool
    reason: str
    estimated_kv_bytes: int


def estimate_kv_bytes(
    total_tokens: int,
    *,
    layers: int = 32,
    kv_heads: int = 8,
    head_dim: int = 128,
    bytes_per_element: int = 2,
) -> int:
    """Estimate K and V cache bytes for one sequence."""
    return 2 * layers * kv_heads * head_dim * bytes_per_element * total_tokens


def check_admission(
    request: RequestPlan,
    snapshot: CapacitySnapshot,
    config: ServingConfig = SERVING_CONFIG,
) -> AdmissionDecision:
    limits = config.capacity
    total_tokens = request.input_tokens + request.max_new_tokens
    estimated_kv_bytes = estimate_kv_bytes(total_tokens)
    usable_kv_bytes = int(
        limits.kv_cache_budget_bytes * (1.0 - limits.kv_cache_reserve_fraction)
    )

    checks = (
        (request.input_tokens > 0, "input_tokens_must_be_positive"),
        (request.max_new_tokens > 0, "max_new_tokens_must_be_positive"),
        (request.input_tokens <= limits.max_input_tokens, "input_token_limit"),
        (request.max_new_tokens <= limits.max_output_tokens, "output_token_limit"),
        (total_tokens <= limits.max_model_tokens, "context_window_limit"),
        (request.remaining_deadline_ms > 0, "deadline_expired"),
        (
            snapshot.active_sequences < limits.max_active_sequences,
            "sequence_capacity",
        ),
        (
            snapshot.scheduled_batch_tokens + total_tokens <= limits.max_batch_tokens,
            "batch_token_capacity",
        ),
        (
            snapshot.kv_cache_bytes_used + estimated_kv_bytes <= usable_kv_bytes,
            "kv_cache_capacity",
        ),
    )

    for passed, reason in checks:
        if not passed:
            return AdmissionDecision(False, reason, estimated_kv_bytes)
    return AdmissionDecision(True, "admitted", estimated_kv_bytes)


EMPTY_CAPACITY = CapacitySnapshot(
    active_sequences=0,
    scheduled_batch_tokens=0,
    kv_cache_bytes_used=0,
)

admission_examples = (
    RequestPlan("normal", input_tokens=512, max_new_tokens=128, remaining_deadline_ms=5000),
    RequestPlan("too-long", input_tokens=4000, max_new_tokens=256, remaining_deadline_ms=5000),
    RequestPlan("expired", input_tokens=128, max_new_tokens=64, remaining_deadline_ms=0),
)

for request_plan in admission_examples:
    decision = check_admission(request_plan, EMPTY_CAPACITY)
    print(
        f"{request_plan.request_id:>8}: accepted={decision.accepted:<5} "
        f"reason={decision.reason:<24} kv={decision.estimated_kv_bytes / 1024**2:.1f} MiB"
    )

In [ ]:
from math import ceil
from statistics import mean


@dataclass(frozen=True)
class RequestMetrics:
    route: str
    status: str
    queue_ms: float
    prefill_ms: float
    ttft_ms: float
    tpot_ms: float
    end_to_end_ms: float
    input_tokens: int
    output_tokens: int
    deadline_ms: int


def percentile(values: Tuple[float, ...], quantile: float) -> float:
    if not values:
        raise ValueError("percentile requires at least one value")
    ordered = sorted(values)
    index = max(0, ceil(quantile * len(ordered)) - 1)
    return ordered[index]


def summarize_request_metrics(
    observations: Tuple[RequestMetrics, ...],
) -> dict[str, float]:
    if not observations:
        raise ValueError("at least one request observation is required")

    successful = tuple(item for item in observations if item.status == "ok")
    deadline_successes = sum(
        item.status == "ok" and item.end_to_end_ms <= item.deadline_ms
        for item in observations
    )
    ttft_values = tuple(item.ttft_ms for item in successful)
    tpot_values = tuple(item.tpot_ms for item in successful)

    return {
        "request_count": float(len(observations)),
        "availability": len(successful) / len(observations),
        "deadline_success_rate": deadline_successes / len(observations),
        "p95_ttft_ms": percentile(ttft_values, 0.95),
        "p95_tpot_ms": percentile(tpot_values, 0.95),
        "mean_queue_ms": mean(item.queue_ms for item in observations),
        "output_tokens_per_request": mean(item.output_tokens for item in successful),
    }


# Synthetic local observations keep this cell deterministic and network-free.
LOCAL_REQUEST_METRICS = (
    RequestMetrics("primary", "ok", 18, 110, 128, 34, 420, 512, 8, 1000),
    RequestMetrics("primary", "ok", 31, 135, 166, 38, 508, 768, 9, 1000),
    RequestMetrics("primary", "ok", 55, 190, 245, 46, 705, 1024, 10, 1000),
    RequestMetrics("primary", "overloaded", 0, 0, 0, 0, 4, 256, 0, 1000),
)

metric_summary = summarize_request_metrics(LOCAL_REQUEST_METRICS)
for metric_name, metric_value in metric_summary.items():
    print(f"{metric_name:>26}: {metric_value:.3f}")

print("\nTelemetry rule: aggregate by bounded labels such as model, route, region, and status.")
print("Do not use request IDs, tenant IDs, prompts, or generated text as metric labels.")

### Guarded smoke, load, and rollout plans

A production smoke test should verify the pinned artifact, readiness state, deterministic local inference, streaming first-token behavior, deadline cancellation, overload rejection, and graceful drain. A load test should increase offered token rate in stages while checking TTFT, TPOT, queue delay, KV-cache pressure, rejection rate, and cost per successful token.

The next cells only build and validate plans from local data. They contain no HTTP client, cloud SDK, subprocess, server startup, or model download. The `RUN_PRODUCTION_*` switches remain `False`; a real executor belongs in an authenticated deployment pipeline with an approved target and rollback owner.

In [ ]:
@dataclass(frozen=True)
class SmokeCheck:
    name: str
    expected_result: str


@dataclass(frozen=True)
class LoadStage:
    name: str
    duration_seconds: int
    offered_requests_per_second: int
    max_concurrency: int
    prompt_tokens: int
    output_tokens: int


LOCAL_SMOKE_PLAN = (
    SmokeCheck("artifact_digest", "loaded revision matches immutable config"),
    SmokeCheck("readiness", "ready only after load, cache allocation, and warm-up"),
    SmokeCheck("short_generation", "local deterministic probe returns expected shape"),
    SmokeCheck("streaming", "first token and completion metrics are recorded"),
    SmokeCheck("deadline", "expired work is cancelled before further decode"),
    SmokeCheck("overload", "capacity exhaustion returns a retryable rejection"),
    SmokeCheck("drain", "readiness clears before active requests finish"),
)

LOCAL_LOAD_PLAN = (
    LoadStage("warm", 60, 2, 4, 256, 64),
    LoadStage("steady", 180, 8, 16, 512, 128),
    LoadStage("target", 180, 16, 32, 512, 128),
    LoadStage("overload", 60, 24, 64, 1024, 256),
    LoadStage("recovery", 120, 4, 8, 256, 64),
)


def validate_local_plans() -> None:
    assert LOCAL_SMOKE_PLAN, "smoke plan must not be empty"
    assert LOCAL_LOAD_PLAN, "load plan must not be empty"
    for stage in LOCAL_LOAD_PLAN:
        assert stage.duration_seconds > 0
        assert stage.offered_requests_per_second > 0
        assert stage.max_concurrency > 0
        assert stage.prompt_tokens + stage.output_tokens <= SERVING_CONFIG.capacity.max_model_tokens


validate_local_plans()
print(f"Validated {len(LOCAL_SMOKE_PLAN)} local smoke checks.")
print(f"Validated {len(LOCAL_LOAD_PLAN)} local load stages.")

if RUN_PRODUCTION_SMOKE or RUN_PRODUCTION_LOAD:
    raise RuntimeError(
        "This notebook defines plans only; production execution must use the approved pipeline."
    )

print("Production smoke/load execution is disabled; no server or network call was made.")

In [ ]:
@dataclass(frozen=True)
class CanaryPolicy:
    traffic_fraction: float
    minimum_availability: float
    max_ttft_regression_fraction: float
    max_tpot_regression_fraction: float


CANARY_POLICY = CanaryPolicy(
    traffic_fraction=0.05,
    minimum_availability=SERVING_CONFIG.slo.availability_target,
    max_ttft_regression_fraction=0.10,
    max_tpot_regression_fraction=0.10,
)


def evaluate_canary(
    baseline: dict[str, float],
    candidate: dict[str, float],
    policy: CanaryPolicy = CANARY_POLICY,
) -> Tuple[str, ...]:
    failures = []
    if candidate["availability"] < policy.minimum_availability:
        failures.append("availability")
    if candidate["p95_ttft_ms"] > baseline["p95_ttft_ms"] * (
        1 + policy.max_ttft_regression_fraction
    ):
        failures.append("p95_ttft")
    if candidate["p95_tpot_ms"] > baseline["p95_tpot_ms"] * (
        1 + policy.max_tpot_regression_fraction
    ):
        failures.append("p95_tpot")
    return tuple(failures)


# Synthetic aggregates stand in for a telemetry query performed by the release pipeline.
BASELINE_WINDOW = {
    "availability": 0.9995,
    "p95_ttft_ms": 720.0,
    "p95_tpot_ms": 52.0,
}
CANDIDATE_WINDOW = {
    "availability": 0.9994,
    "p95_ttft_ms": 760.0,
    "p95_tpot_ms": 54.0,
}

canary_failures = evaluate_canary(BASELINE_WINDOW, CANDIDATE_WINDOW)
rollout_decision = "ROLL BACK" if canary_failures else "PROMOTE"
print(f"Offline canary decision: {rollout_decision}")
print("Failed gates:", canary_failures or "none")

if RUN_PRODUCTION_CANARY:
    raise RuntimeError(
        "Canary traffic changes are intentionally unavailable from this notebook."
    )

print("Production routing is unchanged; RUN_PRODUCTION_CANARY is False.")